# spVIPES: Normalizing Flow vs Gaussian Prior on Perturbation Data

This vignette demonstrates how **spVIPES** decomposes single-cell perturbation
data into **shared** (cell-type identity) and **private** (perturbation-specific)
latent representations. We compare the standard Gaussian prior against a
**Neural Spline Flow (NSF)** prior on the shared latent space.

**Dataset.** IFN-β stimulated PBMCs from the CINEMA-OT example
(Dong *et al.*, *Nature Methods* 2023), originally derived from
Kang *et al.* (2018). The dataset contains ~1 000 cells split between
an unstimulated control and IFN-β treatment, with annotated cell types.

**Goal.** The shared latent should capture cell-type identity (invariant
across conditions), while the private latent should capture
perturbation-specific responses. A normalizing-flow prior can model
multi-modal latent distributions more faithfully than a standard Gaussian,
potentially improving disentanglement.

| Step | What we do |
|---|---|
| §1 | Load the CinemaOT example dataset |
| §2 | Run CinemaOT as a reference decomposition |
| §3 | Prepare data for spVIPES |
| §4 | Train spVIPES with **Gaussian** prior (baseline) |
| §5 | Train spVIPES with **NSF** prior |
| §6 | Compare shared latent spaces (UMAP) |
| §7 | Compare private latent spaces (UMAP) |
| §8 | Quantitative evaluation |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import torch

import pertpy as pt
import spVIPES

np.random.seed(42)
torch.manual_seed(42)
sc.settings.set_figure_params(dpi=100, frameon=False)

## 1. Load the CinemaOT example dataset

`pt.dt.cinemaot_example()` returns ~1 000 PBMCs with two perturbation
conditions (`"No stimulation"` and `"IFNb"`) and cell-type annotations
in `cell_type0528`.

In [ ]:
adata = pt.dt.cinemaot_example()

print(f"Shape           : {adata.shape}")
print(f"Perturbation    : {adata.obs['perturbation'].value_counts().to_dict()}")
print(f"Cell types      : {adata.obs['cell_type0528'].nunique()}")
print()
print(adata.obs['cell_type0528'].value_counts())

In [ ]:
sc.pl.umap(adata, color=["perturbation", "cell_type0528"], wspace=0.5)

## 2. CinemaOT reference decomposition

CINEMA-OT uses optimal transport to separate **confounder** variation
(cell type) from **treatment effects**. We run it here as a reference
point: the confounder embedding is analogous to spVIPES' shared latent,
and the treatment-effect embedding is analogous to the private latent.

In [ ]:
cot = pt.tl.Cinemaot()
sc.pp.pca(adata)
de = cot.causaleffect(
    adata,
    pert_key="perturbation",
    control="No stimulation",
    return_matching=True,
    thres=0.5,
    smoothness=1e-5,
    eps=1e-3,
    solver="Sinkhorn",
    preweight_label="cell_type0528",
)

In [ ]:
# Confounder embedding (analogous to spVIPES shared latent)
sc.pp.neighbors(adata, use_rep="cf")
sc.tl.umap(adata, random_state=0)
adata.obsm["X_umap_cinemaot_cf"] = adata.obsm["X_umap"].copy()

# Treatment-effect embedding (analogous to spVIPES private latent)
sc.pp.neighbors(de, use_rep="X_embedding")
sc.tl.umap(de, random_state=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sc.pl.embedding(adata, basis="umap_cinemaot_cf", color="cell_type0528",
                ax=axes[0], show=False, title="CinemaOT confounder — cell type")
sc.pl.embedding(adata, basis="umap_cinemaot_cf", color="perturbation",
                ax=axes[1], show=False, title="CinemaOT confounder — perturbation")
sc.pl.umap(de, color="cell_type0528",
           ax=axes[2], show=False, title="CinemaOT treatment effect")
plt.tight_layout()
plt.show()

In the confounder space, treated and control cells overlap (confounders
removed), while cell types remain separated — confirming that CINEMA-OT
successfully isolates treatment effects from biological identity.

We now replicate this decomposition with spVIPES and compare the
Gaussian vs NSF prior.

## 3. Prepare data for spVIPES

We split the dataset by `perturbation` into two groups and feed them
to `prepare_adatas`. spVIPES will learn a **shared** latent (capturing
cell-type identity across both conditions) and two **private** latents
(capturing condition-specific variation).

In [ ]:
control = adata[adata.obs["perturbation"] == "No stimulation"].copy()
treated = adata[adata.obs["perturbation"] == "IFNb"].copy()

print(f"Control : {control.shape}")
print(f"Treated : {treated.shape}")

In [ ]:
sdata = spVIPES.data.prepare_adatas({
    "control": control,
    "IFNb": treated,
})

print(f"Combined shape : {sdata.shape}")
print(f"Groups         : {sdata.obs['groups'].value_counts().to_dict()}")

In [ ]:
spVIPES.model.spVIPES.setup_anndata(
    sdata,
    groups_key="groups",
    label_key="cell_type0528",
)

group_indices_list = [
    np.where(sdata.obs["groups"] == g)[0]
    for g in sdata.obs["groups"].unique()
]
print(f"Cells per group: {[len(g) for g in group_indices_list]}")

## 4. Baseline — Gaussian prior

We first train a model with the standard $\mathcal{N}(0, I)$ prior on
both latent spaces.

In [ ]:
N_SHARED   = 10
N_PRIVATE  = 5
N_HIDDEN   = 128
DROPOUT    = 0.1
MAX_EPOCHS = 80
BATCH_SIZE = 128
KL_WARMUP  = 30

torch.manual_seed(42)
np.random.seed(42)

model_gauss = spVIPES.model.spVIPES(
    sdata,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    use_nf_prior=False,
)
print(model_gauss)

In [ ]:
model_gauss.train(
    group_indices_list,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    train_size=0.9,
    early_stopping=False,
    n_epochs_kl_warmup=KL_WARMUP,
)

In [ ]:
plt.plot(model_gauss.history["elbo_train"]["elbo_train"], label="train")
if "elbo_validation" in model_gauss.history:
    plt.plot(model_gauss.history["elbo_validation"]["elbo_validation"], label="val")
plt.xlabel("Epoch")
plt.ylabel("ELBO")
plt.title("Gaussian prior — training")
plt.legend()
plt.show()

## 5. Neural Spline Flow prior

We now train an identical architecture but replace the Gaussian prior on
the **shared** latent with a Neural Spline Flow (NSF). The NSF can model
multi-modal and skewed distributions, which may better capture the
structure of the shared representation.

Under the hood the KL term becomes a Monte Carlo estimate:

$$
\mathrm{KL}\bigl(q(z \mid x) \,\|\, p_\text{flow}(z)\bigr)
\;\approx\;
\log q(z \mid x) - \log p_\text{flow}(z)
$$

The flow parameters are jointly optimised with the VAE — no extra
warm-up schedule is needed.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

model_nf = spVIPES.model.spVIPES(
    sdata,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    use_nf_prior=True,
    nf_type="NSF",
    nf_transforms=3,
    nf_target="shared",
)
print(model_nf)

In [ ]:
model_nf.train(
    group_indices_list,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    train_size=0.9,
    early_stopping=False,
    n_epochs_kl_warmup=KL_WARMUP,
)

In [ ]:
plt.plot(model_nf.history["elbo_train"]["elbo_train"], label="train")
if "elbo_validation" in model_nf.history:
    plt.plot(model_nf.history["elbo_validation"]["elbo_validation"], label="val")
plt.xlabel("Epoch")
plt.ylabel("ELBO")
plt.title("NSF prior — training")
plt.legend()
plt.show()

## 5b. NSF prior + Disentanglement objective

spVIPES supports an optional **disentanglement objective** (inspired by
CellDISECT and Multi-ContrastiveVAE) that enforces:

* `z_shared` encodes cell-type biology but **not** perturbation status
* `z_private` encodes perturbation-specific variation but **not** cell-type biology

We train a third model with the NSF prior **and** the full disentanglement
preset, so we can ask whether explicit disentanglement adds value on top of
an already-flexible flow prior. See `disentangle_ablation.ipynb` for a
per-component ablation.


In [ ]:
torch.manual_seed(42)
np.random.seed(42)

model_nf_dis = spVIPES.model.spVIPES(
    sdata,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    use_nf_prior=True,
    nf_type="NSF",
    nf_transforms=3,
    nf_target="shared",
    disentangle_preset="full",
)
model_nf_dis.train(
    group_indices_list,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    train_size=0.9,
    early_stopping=False,
    n_epochs_kl_warmup=KL_WARMUP,
)


## 6. Compare shared latent spaces

The **shared** latent should capture cell-type identity while mixing
cells from both perturbation conditions. Good integration means:

- Cell types are well-separated (high biological conservation)
- Perturbation conditions are well-mixed (high batch mixing)

In [ ]:
latents_gauss = model_gauss.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)
latents_nf = model_nf.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)
latents_nf_dis = model_nf_dis.get_latent_representation(group_indices_list, batch_size=BATCH_SIZE)

print("Latent keys:", list(latents_gauss.keys()))


In [ ]:
def stitch_shared(sdata, latents):
    """Place per-group shared latents back into the combined obs order."""
    n_obs = sdata.n_obs
    dim = latents["shared_reordered"][0].shape[1]
    out = np.zeros((n_obs, dim), dtype=np.float32)
    for gi, g_positions in enumerate(sdata.uns["groups_obs_indices"]):
        out[g_positions] = latents["shared_reordered"][gi]
    return out

sdata.obsm["X_shared_gauss"]   = stitch_shared(sdata, latents_gauss)
sdata.obsm["X_shared_nf"]      = stitch_shared(sdata, latents_nf)
sdata.obsm["X_shared_nf_dis"]  = stitch_shared(sdata, latents_nf_dis)

for rep, key_added in [
    ("X_shared_gauss", "gauss"),
    ("X_shared_nf", "nf"),
    ("X_shared_nf_dis", "nf_dis"),
]:
    sc.pp.neighbors(sdata, use_rep=rep, key_added=key_added, n_neighbors=15)
    sc.tl.umap(sdata, neighbors_key=key_added, random_state=0)
    sdata.obsm[f"umap_{key_added}"] = sdata.obsm["X_umap"].copy()


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
for row, (key_added, title) in enumerate([
    ("gauss", "Gaussian"),
    ("nf", "NSF"),
    ("nf_dis", "NSF + disentangle=full"),
]):
    sc.pl.embedding(sdata, basis=f"umap_{key_added}", color="cell_type0528",
                    ax=axes[row, 0], show=False,
                    title=f"{title} shared — cell type")
    sc.pl.embedding(sdata, basis=f"umap_{key_added}", color="perturbation",
                    ax=axes[row, 1], show=False,
                    title=f"{title} shared — perturbation")
plt.tight_layout(); plt.show()


## 7. Compare private latent spaces

The **private** latent for the IFN-β group should capture
perturbation-specific gene programs — variation that is *not* shared
with the unstimulated control. Ideally, cell-type structure should be
absent (or much weaker) in the private space, since cell identity is
a shared confounder.

In [ ]:
groups = list(sdata.obs["groups"].unique())

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, (latents, label) in enumerate([(latents_gauss, "Gaussian"),
                                         (latents_nf, "NSF")]):
    for col, gi in enumerate(range(len(groups))):
        g_positions = sdata.uns["groups_obs_indices"][gi]
        g_obs = sdata.obs.iloc[g_positions].copy()

        priv = ad.AnnData(
            X=latents["private_reordered"][gi],
            obs=g_obs,
        )
        sc.pp.neighbors(priv, use_rep="X", n_neighbors=15)
        sc.tl.umap(priv, random_state=0)

        sc.pl.umap(priv, color="cell_type0528", ax=axes[row, col],
                   show=False,
                   title=f"{label} private — {groups[gi]}")

plt.tight_layout()
plt.show()

If the model successfully disentangles, the private UMAPs should show
**weaker** cell-type clustering than the shared UMAPs — cell identity is
pushed to the shared space, while perturbation-specific variation stays
private.

## 8. Quantitative comparison

We report two complementary metrics on the **shared** latent:

| Metric | Measures | Good value |
|---|---|---|
| **ARI** (Adjusted Rand Index) | Biological conservation — do Leiden clusters match cell types? | Higher ↑ |
| **Batch mixing entropy** | Perturbation mixing — are conditions mixed in local neighbourhoods? | Higher ↑ |

A successful shared latent achieves **high ARI and high batch mixing
simultaneously**.

In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors


def leiden_ari(rep, labels, resolution=0.8):
    tmp = ad.AnnData(X=rep)
    sc.pp.neighbors(tmp, use_rep="X", n_neighbors=15)
    sc.tl.leiden(tmp, resolution=resolution, random_state=0)
    return adjusted_rand_score(labels, tmp.obs["leiden"].values)


def batch_entropy(rep, batch_labels, k=30):
    batch_labels = np.asarray(batch_labels)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(rep)
    _, idx = nn.kneighbors(rep)
    idx = idx[:, 1:]
    n_batches = len(np.unique(batch_labels))
    max_entropy = np.log2(n_batches)
    entropies = []
    for i in range(rep.shape[0]):
        neigh_batches = batch_labels[idx[i]]
        _, counts = np.unique(neigh_batches, return_counts=True)
        p = counts / counts.sum()
        H = -np.sum(p * np.log2(p + 1e-12))
        entropies.append(H / max_entropy)
    return float(np.mean(entropies))


cell_types = sdata.obs["cell_type0528"].values
perturbation = sdata.obs["groups"].values

results = []
for name, key in [("Gaussian", "X_shared_gauss"), ("NSF", "X_shared_nf")]:
    rep = sdata.obsm[key]
    ari = leiden_ari(rep, cell_types)
    mix = batch_entropy(rep, perturbation, k=30)
    results.append({"prior": name, "ARI (cell types)": ari, "batch mixing": mix})

results_df = pd.DataFrame(results).set_index("prior")
results_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(results_df))
width = 0.35
ax.bar(x - width / 2, results_df["ARI (cell types)"],  width, label="ARI (cell types) ↑")
ax.bar(x + width / 2, results_df["batch mixing"],      width, label="batch mixing ↑")
ax.set_xticks(x)
ax.set_xticklabels(results_df.index)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right", fontsize=9)
ax.set_title("Shared-latent quality: Gaussian vs NSF prior")
plt.tight_layout()
plt.show()

## Summary

We used the CINEMA-OT perturbation dataset to compare two spVIPES
configurations on the task of disentangling **cell identity** (shared)
from **IFN-β treatment effects** (private).

| | Gaussian prior | NSF prior |
|---|---|---|
| **Shared latent prior** | $\mathcal{N}(0, I)$ | Neural Spline Flow |
| **KL computation** | Closed-form | Monte Carlo |
| **Extra parameters** | — | Flow transforms |

**Key takeaways:**

- Both models successfully decompose the data into shared (cell type)
  and private (perturbation) latent spaces, comparable to the CinemaOT
  confounder / treatment-effect decomposition.
- The NSF prior gives the shared latent more flexibility to model
  non-Gaussian structure, which can improve biological conservation
  and batch mixing — especially on larger datasets and higher
  dimensional latent spaces.
- For a thorough benchmark, scale up to the full dataset and increase
  `MAX_EPOCHS`.

### References

- Novella-Rausell, C., Peters, D.J.M., & Mahfouz, A. (2023).
  *Integrative learning of disentangled representations in multi-view
  single-cell data.* bioRxiv.
- Dong, M. *et al.* (2023). *Causal identification of single-cell
  experimental perturbation effects with CINEMA-OT.* Nature Methods.
- Durkan, C. *et al.* (2019). *Neural Spline Flows.* NeurIPS.